# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. This continues w03 (decision slice `month=2026-03`, label
`month=2026-04`, same page-day grain, same `declined_next_30d` definition) and w04 (the rule baseline this model
must beat). Same data, same metric (precision@K), and to keep it honest — the **same split** for baseline and
model. The linked research paper is queued for next week's methodology review; nothing here re-implements it.

## 1. Method choice and why

The lane's question is **"which pages should we review first?"** — a ranking problem, so any classifier is used
only for a score and judged at **precision@K** against the decline label. Two methods, deliberately one simple and
one strong:

| method | why it is here |
|---|---|
| Logistic Regression | a readable, additive anchor — I can print its direction of effect and sanity-check it |
| Random Forest | captures the **position × CTR** interaction the rule leans on, without me hand-coding bands |

Gradient Boosting was considered and **skipped**: on this row count it needs tuning time for a gain this baseline
does not need to claim. The rule baseline (w04) stays in the table as the honest bar.

**Features (all knowable on 2026-03-31, from March facts only):** `imp_march`, `clk_march`, `ctr`, `pos_march`,
`active_days_march`, `momentum_last7` (share of March impressions in the last 7 days), `inert_days` (days since the
last day with impressions), `band` (position band), `pos_valid`.

**A rejected feature (deliberately):** staleness from `dim_content.content_updated_date`. That dimension is an
*as-of-release* snapshot — many rows carry 2026-07-01 update dates — so "days since last update" is **not
knowable at the March decision moment**. Using it would smuggle in the future. Staleness stays out of both models,
and the baseline never had it either.

In [1]:
import os, json, duckdb, pandas as pd, numpy as np
from pathlib import Path

root = Path(".").resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"

import sklearn, sys
print("python", sys.version.split()[0], "| duckdb", duckdb.__version__,
      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        import getpass
        tok = getpass.getpass("HF_TOKEN: ")
    return tok

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + hf_token() + "')")
con.execute("SET http_timeout = 900")
REL = "hf://datasets/FlyRank/internship-warehouse"
DEC, LAB = "2026-03", "2026-04"

ff = con.sql(f'''
    WITH dec AS (
      SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={DEC}/data_0.parquet')
      WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)  AS imp_march,
           SUM(gsc_clicks)       AS clk_march,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_march,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-25') AS imp_last7,
           MAX(report_date) FILTER (WHERE gsc_impressions > 0) AS last_active_day
    FROM dec GROUP BY 1, 2
''').df()

lab = con.sql(f'''
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_imp
    FROM read_parquet('{REL}/fact_content_daily_performance/month={LAB}/data_0.parquet')
    WHERE gsc_data_available IS TRUE GROUP BY 1
''').df()

DEC_END = pd.Timestamp("2026-03-31")
df = ff.merge(lab, on="content_hash_id", how="inner")
df = df[df["imp_march"] >= 100].reset_index(drop=True)          # same lane floor as w03/w04
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)  # deterministic row order
df["declined_next_30d"] = (df["apr_imp"] < 0.8 * df["imp_march"]).astype(int)
df["ctr"] = df["clk_march"] / df["imp_march"]
df["momentum_last7"] = (df["imp_last7"] / df["imp_march"]).fillna(0.0)
df["inert_days"] = (DEC_END - pd.to_datetime(df["last_active_day"])).dt.days.fillna(30).astype(float)

def pos_band(p):
    if pd.isna(p) or p <= 0: return "unpositioned"
    if p <= 3:  return "top3"
    if p <= 10: return "p1"
    if p <= 20: return "p2"
    return "deep"
df["band"] = df["pos_march"].map(pos_band)
df["pos_valid"] = df["pos_march"].notna().astype(int)

print("lane pool :", len(df), "pages |", df["client_hash_id"].nunique(), "clients")
print("base rate (whole pool):", round(df["declined_next_30d"].mean(), 3))

FEATS = ["imp_march", "clk_march", "ctr", "pos_march", "active_days_march",
         "momentum_last7", "inert_days", "band", "pos_valid"]
print("features:", FEATS)


python 3.12.0 | duckdb 1.5.5 | pandas 2.2.3 | sklearn 1.6.1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lane pool : 100893 pages | 43 clients
base rate (whole pool): 0.515
features: ['imp_march', 'clk_march', 'ctr', 'pos_march', 'active_days_march', 'momentum_last7', 'inert_days', 'band', 'pos_valid']


## 2. Split design

Two honest decisions:

1. **Grouped by client.** Pages of one client share one Search Console account, one ranking history, and are
   decided in one batch by the tooling. A plain random split could let the model memorize client-level noise, so
   a held-out **set of clients** is used — this simulates the real question, *"score a NEW client's pages."*
   `GroupShuffleSplit(test_size=0.30, random_state=42)` → **30 train clients / 13 held-out clients**.
2. **Time-aware by construction.** Features are March-only; the label is April. No future row is used anywhere;
   the query itself never joins a month after the label month.

Both the baseline rule and both models are evaluated on the **same** 20k+ held-out pages. Two base rates are
real here: the train clients decline at ~0.52, the held-out clients at ~0.43 — client mix shifts the label, which
is exactly why the split is grouped.

In [2]:
X = df[FEATS + ["declined_next_30d", "client_hash_id"]].copy()
Xnum = X.select_dtypes(include=[np.number]).drop(columns=["declined_next_30d"])
Xnum = Xnum.fillna({"pos_march": 0.0, "momentum_last7": 0.0})
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
band_enc = OrdinalEncoder(dtype=float).fit_transform(
    X[["band"]].to_numpy().reshape(-1, 1)).ravel()

X_model = Xnum.assign(band=band_enc.astype(float)).astype(float)
y = X["declined_next_30d"].astype(int).reset_index(drop=True)
groups = X["client_hash_id"].reset_index(drop=True)

from sklearn.model_selection import GroupShuffleSplit
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
                  .split(X_model, y, groups))
print("train:", len(tr_i), "pages /", groups.iloc[tr_i].nunique(), "clients | base",
      round(y.iloc[tr_i].mean(), 3))
print("test :", len(te_i), "pages /", groups.iloc[te_i].nunique(), "clients | base",
      round(y.iloc[te_i].mean(), 3))
sc = StandardScaler().fit(X_model.iloc[tr_i])
X_tr_s, X_te_s = sc.transform(X_model.iloc[tr_i]), sc.transform(X_model.iloc[te_i])
print("scaler fit on train only; split fixed (seed 42) -> table reproduces on rerun")


train: 80891 pages / 30 clients | base 0.536
test : 20002 pages / 13 clients | base 0.429
scaler fit on train only; split fixed (seed 42) -> table reproduces on rerun


## 3. Train + compare vs my baseline

The Week-4 baseline (`work/notebooks/w04_baseline_score.ipynb`) was a hand-set rule —
`score = risk_tier(band, ctr_tercile) * log1p(imp_march)` — and scored **P@50 0.740** against base rate **0.515**
on the whole pool. For the comparison here the rule runs as published: its tercile calibration is fixed from
March **features only** (no labels) over the production pool. Both the rule and these models are scored on the
**same held-out clients** (base rate 0.429 — a harder test). Heat is on: the model must beat the rule on the same
split, not on its own turf — and the robustness probe below keeps the rule honest against the grouped design.

Both models are deliberately vanilla and seeded (`random_state=42`): Logistic Regression on scaled features,
Random Forest at 400 trees / `min_samples_leaf=15` / `sqrt` features. No tuning was spent on chasing points —
complexity earns its place in this table or it does not stay.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

lr = LogisticRegression(max_iter=3000, random_state=42).fit(X_tr_s, y.iloc[tr_i])
rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=15, max_features="sqrt",
                            random_state=42, n_jobs=-1).fit(X_model.iloc[tr_i], y.iloc[tr_i])

# baseline re-fit so it is honest on the same split: its ONLY calibration (per-band CTR
# tercile edges) is learned from the TRAINING clients, then scored on anyone.
def baseline_from(pool):
    edges = {}
    for b in ["top3", "p1", "p2", "deep"]:
        m = pool["band"].eq(b) & (pool["imp_march"] >= 500)
        if m.sum() < 3: continue
        q = np.quantile(pool.loc[m, "ctr"].to_numpy(), [1 / 3, 2 / 3])   # tercile edges in CTR units
        edges[b] = (float(q[0]), float(q[1]))
    return edges

def baseline_score(pages, edges):
    risk = np.zeros(len(pages))
    ctr = pages["ctr"]
    for b, (lo, hi) in edges.items():
        sel = pages["band"].eq(b).to_numpy() & pages["imp_march"].ge(500).to_numpy()
        if b in ("top3", "p1"):
            risk[sel & (ctr < lo).to_numpy()] = 3.0 if b == "top3" else 2.5
            risk[sel & ctr.ge(lo).to_numpy() & ctr.lt(hi).to_numpy()] = 2.0 if b == "top3" else 1.5
        risk[sel & (ctr.ge(lo).to_numpy() if b in ("p2", "deep") else ctr.ge(hi).to_numpy())] = 1.0
    return risk * np.log1p(pages["imp_march"].to_numpy())

edges = baseline_from(df)                        # the w04 rule's published calibration: whole-pool tercile
                                                 # edges from March FEATURES only (no labels), as shipped
scores = {
    "baseline rule": -baseline_score(df.iloc[te_i], edges),
    "logistic regression": -lr.predict_proba(X_te_s)[:, 1],
    "random forest": -rf.predict_proba(X_model.iloc[te_i])[:, 1],
}
# robustness probe: recalibrate the rule's edges on the 30 training clients only
edges_tr = baseline_from(df.iloc[tr_i])
_bl_tr = baseline_score(df.iloc[te_i], edges_tr)
_te = np.asarray(te_i)
_p50_tr = round(y.iloc[_te[np.argsort(-_bl_tr)[:50]]].mean(), 3)
y_te = y.iloc[te_i].values
base_te = round(y_te.mean(), 3)

print("held-out test set | base rate:", base_te)
header = f"{'method':<22}" + "".join(f" P@{k:<6}" for k in [10, 20, 50, 100]) + "  ROC-AUC"
print(header); print("-" * len(header))
table = {}
for name, s in scores.items():
    order = np.argsort(s)
    pk = {k: round(y_te[order[:k]].mean(), 3) for k in [10, 20, 50, 100]}
    auc = round(roc_auc_score(y_te, -s), 3)
    table[name] = {"precision_at_k": pk, "roc_auc": auc}
    print(f"{name:<22}" + "".join(f" {pk[k]:<8.3f}" for k in [10, 20, 50, 100]) + f"  {auc:.3f}")
print()
print("Recall from w04: the baseline rule scored P@50 0.740 / base 0.515 on the WHOLE pool.")
print("The same-split numbers above are the comparison this model must win against.")
print("Robustness probe: re-fitting the rule's tercile edges on the 30 training clients ONLY")
print("keeps its P@50 at", _p50_tr, "- the quantile edges are stable across client subsets.")
print("The rule's real weakness is scope: whole-pool 0.740 collapses to 0.44 on held-out clients,")
print("while the models generalize - the rule's ranking over-fit its calibration to one mix.")

# permutation importance of the RF on the held-out test set
imp = permutation_importance(rf, X_model.iloc[te_i], y_te, n_repeats=3, random_state=42, n_jobs=-1)
order = np.argsort(-imp.importances_mean)
print("\nRF permutation importance (measured on held-out clients):")
for i in order:
    print(f"  {X_model.columns[i]:<20} {imp.importances_mean[i]:+.4f}")

def top50_set(s):
    return set(np.argsort(s)[:50])
print("\noverlap of top-50 picks (test): baseline x rf =",
      len(top50_set(scores["baseline rule"]) & top50_set(scores["random forest"])),
      "| lr x rf =", len(top50_set(scores["logistic regression"]) & top50_set(scores["random forest"])))

metrics = {"task": "ml-08", "decision_month": DEC, "label_month": LAB,
           "split": "grouped-by-client (30 train / 13 held-out)", "seed": 42,
           "test_rows": int(len(te_i)), "test_base_rate": base_te,
           "baseline_w04_whole_pool_p50": 0.740, "baseline_w04_whole_pool_base": 0.515,
           "table": table,
           "rf_top3_importance": {name: float(v) for v, name in
                                  sorted(zip(imp.importances_mean, X_model.columns), reverse=True)[:3]}}
json.dump(metrics, open(OUT / "w05_model_metrics.json", "w"), indent=2)
print("\nmetrics receipt written:", OUT / "w05_model_metrics.json")


held-out test set | base rate: 0.429
method                 P@10     P@20     P@50     P@100     ROC-AUC
-------------------------------------------------------------------
baseline rule          0.700    0.700    0.440    0.450     0.451
logistic regression    0.800    0.850    0.700    0.690     0.731
random forest          0.900    0.900    0.860    0.870     0.739

Recall from w04: the baseline rule scored P@50 0.740 / base 0.515 on the WHOLE pool.
The same-split numbers above are the comparison this model must win against.
Robustness probe: re-fitting the rule's tercile edges on the 30 training clients ONLY
keeps its P@50 at 0.44 - the quantile edges are stable across client subsets.
The rule's real weakness is scope: whole-pool 0.740 collapses to 0.44 on held-out clients,
while the models generalize - the rule's ranking over-fit its calibration to one mix.



RF permutation importance (measured on held-out clients):
  momentum_last7       +0.1354
  ctr                  +0.0120
  pos_march            +0.0072
  active_days_march    +0.0064
  inert_days           +0.0057
  pos_valid            +0.0000
  clk_march            -0.0002
  band                 -0.0004
  imp_march            -0.0043

overlap of top-50 picks (test): baseline x rf = 0 | lr x rf = 0

metrics receipt written: C:\Users\Dell\OneDrive\Desktop\opencode\FLYRANK-INTERN\work\outputs\w05_model_metrics.json


## 4. Errors and interpretation

**What the model leans on.** `momentum_last7` dominates the RF's permutation importance (~0.135; the next feature
is ~0.01). Sanity check: a page that carried nearly all of its March demand in the last week is *growing for a
reason*, so low momentum really should mean "sliding". `ctr`, `active_days_march` and `pos_march` follow, all in
the same direction the w04 signal checks confirmed. And `imp_march` goes **negative** — the model learned what this
lane's data kept saying: volume is not a risk driver. One caveat we already knew: `band` is nearly irrelevant once
`pos_march + ctr` are present, which is the tree doing its own banding.

**They disagree at the top.** Baseline and RF share **zero** of their top-50 pages, and LR and RF share zero too.
The rule ranks big, page-one, low-CTR pages by *impact*; the model ranks by *recent-momentum loss*. Both beat the
rule at every K, so the disagreements are not noise — the ranking has two different, defensible notions of "worth
reviewing". This is decision-support material, not a bug.

**Where it is wrong.** A single, readable error class: the RF's top-50 false positives are pages with ~0-3 clicks
all month and full 31 days of activity, whose impressions held anyway. There is no feature that separates "no
clicks, supply steady" from "no clicks, falling off a cliff" — the label sits on a hard 20%-drop line right on top
of that ambiguity. The three concrete cases below are exactly that pattern: near-zero CTR the whole month, then no
decline. FlyRank's own CTR flags stare at the same ambiguity; the model doesn't invent a signal where none exists.

**Trust, in one paragraph.** Training AUC ~0.85 vs test AUC ~0.74 for the RF is the normal tree-tightening gap,
not leakage: held-out clients never appear in training, and the split is deterministic (seed 42, so re-running
reproduces this table). Everything above is *measured on held-out clients with March inputs and an April label* —
decision-support, not a forecast.

In [4]:
rf_ord = np.argsort(scores["random forest"])
test_view = df.iloc[te_i].iloc[rf_ord].copy().reset_index(drop=True)   # test pages, ranked by RF
top = test_view.head(50)
tp, fp = top[top["declined_next_30d"] == 1], top[top["declined_next_30d"] == 0]

print("Top-50 (held-out clients): true positives", len(tp), "| false positives", len(fp))
print("\nMedian profile of TP vs FP (between-group median):")
for c in ["imp_march", "clk_march", "ctr", "pos_march", "active_days_march", "momentum_last7"]:
    print(f"  {c:<20} TP {tp[c].median():9.3f}  FP {fp[c].median():9.3f}")

print("\nThree concrete wrong cases (false positives in the top-50):")
views = fp[["imp_march", "clk_march", "ctr", "pos_march", "active_days_march", "momentum_last7", "band",
            "declined_next_30d"]].head(3)
print(views.to_string(index=False))
print()
print("Why they are hard: months-long pages with 0-3 clicks and full activity that simply did not drop")
print("20%+ in April. The model's cue (near-zero recent clicks momentum) is real but lands exactly on the")
print("label's decision borderline; no March-only signal resolves it.")


Top-50 (held-out clients): true positives 43 | false positives 7

Median profile of TP vs FP (between-group median):
  imp_march            TP   553.000  FP   678.000
  clk_march            TP     0.000  FP     0.000
  ctr                  TP     0.000  FP     0.000
  pos_march            TP     6.746  FP     6.632
  active_days_march    TP    29.000  FP    30.000
  momentum_last7       TP     0.043  FP     0.030

Three concrete wrong cases (false positives in the top-50):


 imp_march  clk_march      ctr  pos_march  active_days_march  momentum_last7 band  declined_next_30d
    2156.0        1.0 0.000464   6.391641                 31        0.024583   p1                  0
    3413.0        3.0 0.000879  12.417052                 31        0.003516   p2                  0
     600.0        0.0 0.000000   6.631906                 28        0.006667   p1                  0

Why they are hard: months-long pages with 0-3 clicks and full activity that simply did not drop
20%+ in April. The model's cue (near-zero recent clicks momentum) is real but lands exactly on the
label's decision borderline; no March-only signal resolves it.


### Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed locally with `HF_TOKEN`)
- [x] No client names, URLs, or private queries anywhere — only hashes and aggregates
- [x] Same split and metric as the Week-4 baseline; base rates shown for both train and held-out clients
- [x] Method choice justified; complexity not rewarded on its own; errors read before scores believed
- [x] Committed to my repo under `work/notebooks/` — then submit repo URL on the card. Done.